# Cognopolis · Урок M2 — боевой цикл с отступлением

Ты управляешь жителем игры **Cognopolis** не мышкой, а **кодом-агентом**. В уроке M1 житель мирно собирал ресурсы — теперь в мире завелись **мобы**. Тот же цикл `observe → decide → act → wait` обрастает **условием безопасности**: перед стычкой агент смотрит на свой hp и на силу врага и решает — **драться, отойти и отдохнуть, или вовсе не лезть**.

**Что построим.** Реактивного бойца: он находит врага, дерётся со слабым (**гоблином**), **отдыхает, когда мало hp**, и **не суётся** на сильного (**волка**), пока не окрепнет. Перестаёт умирать в первой же стычке.

**Тир агента:** реактивный+ (майлстоун игры **M2**). Тот же реактивный цикл, что в M1, плюс ветвление по hp и силе цели. **Дальше по курсу:** планировщик (M3), LLM-агент (M4).

> ⚙️ **Working-first.** Ноутбук рассчитан на прогон `Run all` без правок — нужно лишь задать `BASE_URL` живого мира и свой `COGNOPOLIS_TOKEN` (из Ратуши). Учебная активность — в секции **«Задачи»**. Базовый агент уже безопасно фармит гоблина; твоё дело — сделать его умнее.

**Ссылки** (подставь адрес своего мира вместо `<BASE_URL>`):
- API-доки (Swagger, кнопка **Authorize**): `<BASE_URL>/docs`
- 👀 Смотреть за своим агентом в браузере: `<BASE_URL>/?token=<твой токен>` (read-only)
- Контракт: житель видит мир **только** через API (`cognopolis_client`).
- Раньше этого — пройди **реактивного сборщика (M1)**.

## 1. Сетап

Ставим официальный клиент игры из git-URL (PyPI пока нет) и задаём адрес мира.

In [ ]:
%pip install -q "cognopolis-client @ git+https://github.com/ITrubnikov/Train_of_Thought-Cognopolis.git#subdirectory=client"

In [ ]:
import os
from cognopolis_client import Client, GameError

# ⬇️ ЖИВОЙ МИР COGNOPOLIS (публичный инстанс). Можно переопределить переменной COGNOPOLIS_URL.
BASE_URL = os.environ.get("COGNOPOLIS_URL", "https://kindomklaster.com")

# ⬇️ ТВОЙ ТОКЕН ДОСТУПА (полный доступ к твоему жителю).
#   1. Открой BASE_URL в браузере и зарегистрируйся (логин + пароль).
#   2. Ратуша → раздел «ваш аккаунт» → кнопка «копировать» — это твой токен.
#   3. Вставь его в COGNOPOLIS_TOKEN ниже (или задай переменную окружения / секрет Colab/Kaggle).
TOKEN = os.environ.get("COGNOPOLIS_TOKEN", "")  # ← вставь токен в кавычки, если не используешь env
assert TOKEN, f"Вставь токен: зарегистрируйся на {BASE_URL}, скопируй токен из Ратуши и задай COGNOPOLIS_TOKEN."

c = Client(BASE_URL, token=TOKEN)

# Мягкая проверка связи: если мир недоступен — не пугаем трейсбеком, живые ячейки ниже пропустятся.
WORLD_UP = True
try:
    Client(BASE_URL).get_map()  # GET /map не требует токена
except Exception as e:
    WORLD_UP = False
    print(f"⚠️  Мир {BASE_URL} недоступен ({type(e).__name__}). Живые ячейки пропущу — проверь COGNOPOLIS_URL или попробуй позже.")

print("Мир:", BASE_URL, "| на связи:", WORLD_UP)
# Тир M2 — реактивный+: LLM не нужен (он появится на M4).

## 2. Разогрев — смотрим на себя и на врагов

Токеном агент и действует, и по нему же можно наблюдать за жителем в браузере. `get_character()` даёт твой **hp** — главный новый показатель урока, а `get_map()["enemies"]` — список мобов с их силой. Это «глаза» бойца: жив ли враг, решаем по полю `alive`, а не по картинке клетки.

In [ ]:
if WORLD_UP:
    print("👀 Смотри за жителем в браузере:", f"{BASE_URL}/?token={TOKEN}")

    ch = c.get_character()
    print("позиция:", (ch["x"], ch["y"]), "| hp:", f'{ch["hp"]}/{ch["max_hp"]}', "| recovering:", ch["recovering"])

    for e in c.get_map()["enemies"]:
        print("  враг:", e["kind"], "в", (e["x"], e["y"]), "| жив:", e["alive"], "| hp:", f'{e["hp"]}/{e["max_hp"]}')

    # Одно действие: шаг в сторону. В ответе — новое состояние и cooldown (естественный ритм петли).
    res = c.move(ch["x"] + 1, ch["y"], reason="разогрев — пробую сходить")
    print("cooldown:", res["cooldown"], "c | новая позиция:", (res["character"]["x"], res["character"]["y"]))
    c.wait_cooldown()  # observe → decide → act → ВОТ ЭТО ОЖИДАНИЕ

Бой решается **одним вызовом** `c.fight()` — он проигрывает всю стычку и возвращает `combat_log`:

- `outcome` — `"win"` или `"death"` (ретрита больше нет: бой идёт до конца);
- `rounds` — список раундов (удар игрока / ответ врага по hp) для разбора;
- `xp_gained`, `loot`, `recovering`.

Управлять hp **между** ударами внутри одного боя нельзя — поэтому решать «драться или нет» нужно **до** вызова `fight()`. Это и есть весь урок M2.

## 3. Разбор — паттерн реактивного+ агента

Тот же цикл, что в M1, но `decide` теперь ветвится по состоянию **себя и врага**:

```
observe (hp + enemies)  →  decide (rest / fight / обойти)  →  act  →  wait  →  (снова observe)
```

- **observe** — `get_character()` (мой hp, recovering) + `get_map()["enemies"]` (кто жив и насколько силён);
- **decide** — правила безопасности:
  - мало hp → **rest** (отойти, восстановиться);
  - рядом враг по силам → **fight**;
  - враг слишком силён (волк) → **обойти**, не лезть;
- **act** — одно действие: `rest` / `move` (шаг к цели) / `fight`;
- **wait** — `wait_cooldown()`, и цикл повторяется.

Ключевая мысль M2: **смотри перед тем, как бить**. Смерть стоит дорого (потеря рюкзака + возврат домой) — детальный разбор и трейс боя с волком в лекции урока.

## 4. Задачи — собери реактивного бойца

Ниже — **рабочий каркас**: хелперы + `decide()` + петля `run()`. Базовая версия уже безопасно фармит ближайшего врага: она стартует от дома и делает всего несколько ходов (`run(rounds=8)`), поэтому не успевает попасть в беду. Но она «туповата»: **не бережёт hp** и **бьёт любого ближайшего врага в лоб**. Сними тренировочные колёсики — подними `rounds` — и базовый агент, добив гоблина, побредёт к **волку** и **погибнет** (а это потеря рюкзака и возврат домой).

**Твоя задача — сделать его безопасным на любом прогоне:**

1. **hp-guard.** Если hp ниже порога (`HP_REST_THRESHOLD`) — не лезь в бой, верни действие `"rest"`. Тогда длинный фарм не закончится гибелью.
2. **Выбор цели.** Вместо «ближайший любой враг» используй правило `is_safe_target(e, ch)` — бейся только с тем, кого реально одолеешь (волка пропускай, пока слаб). После доработки агент, добив гоблина, будет отдыхать, а не идти на волка.

Подсказки помечены `# TODO`. Ноутбук исполняется и до, и после правок — улучшай постепенно.

In [ ]:
def manhattan(ax, ay, bx, by):
    return abs(ax - bx) + abs(ay - by)

def nearest_living(ch, enemies):
    """Ближайший ЖИВОЙ враг (по манхэттену). Живой — по полю e["alive"], не по картинке клетки."""
    alive = [e for e in enemies if e["alive"]]
    if not alive:
        return None
    return min(alive, key=lambda e: manhattan(ch["x"], ch["y"], e["x"], e["y"]))

def adjacent_xy(ch, tx, ty):
    """Можно действовать вплотную: на той же клетке или рядом."""
    return manhattan(ch["x"], ch["y"], tx, ty) <= 1

def step_toward(ch, tx, ty):
    """Один ортогональный шаг к цели: сперва по X, потом по Y (карта без стен)."""
    if ch["x"] != tx:
        return ch["x"] + (1 if tx > ch["x"] else -1), ch["y"]
    return ch["x"], ch["y"] + (1 if ty > ch["y"] else -1)

def go_home(max_steps=14):
    """Тренировочные колёсики: вернуться домой (0, 0), чтобы демо было предсказуемым и безопасным."""
    for _ in range(max_steps):
        ch = c.get_character()
        if (ch["x"], ch["y"]) == (0, 0):
            return
        c.move(*step_toward(ch, 0, 0), reason="возвращаюсь домой перед вылазкой")
        c.wait_cooldown()

# Готовые инструменты для «Задач» (базовый decide() их пока НЕ использует):
HP_REST_THRESHOLD = 8  # ниже этого hp лучше отдохнуть, чем лезть в бой

def is_safe_target(e, ch):
    """Безопасная ли цель? Грубое правило: бьёмся только с заметно более слабым врагом.
    Гоблин (max_hp 8) по силам, волк (max_hp 14) — пока нет."""
    return e["max_hp"] * 2 <= ch["max_hp"]

In [ ]:
def decide(ch, world):
    """Верни (target_x, target_y, action, reason). action: "fight" или "rest".

    БАЗОВАЯ версия (работает, но туповата): идёт к ближайшему ЖИВОМУ врагу и бьёт в лоб —
    кто бы это ни был. Не смотрит на hp и не разбирает, по силам ли враг. Доработай по «Задачам».
    """
    # TODO 1 — hp-guard: если мало hp, отдохни на месте вместо боя:
    #   if ch["hp"] < HP_REST_THRESHOLD:
    #       return ch["x"], ch["y"], "rest", "мало hp — отдыхаю"

    targets = world["enemies"]
    # TODO 2 — выбор цели: вместо «любой враг» бери только безопасные цели (волка пропусти):
    #   targets = [e for e in world["enemies"] if is_safe_target(e, ch)]

    enemy = nearest_living(ch, targets)
    if enemy is None:
        return ch["x"], ch["y"], "rest", "живых врагов нет — перевожу дух"
    return enemy["x"], enemy["y"], "fight", f'иду на {enemy["kind"]} (базовая версия — улучшь меня!)'


def run(rounds=8):
    go_home()  # стартуем от дома (тренировочные колёсики безопасности)
    for _ in range(rounds):
        ch = c.get_character()                       # observe (себя)
        world = c.get_map()                          # observe (мир: враги меняются)
        tx, ty, action, reason = decide(ch, world)   # decide
        if action == "rest":                         # act: отдых
            c.rest(reason=reason)
            print("  отдых — hp восстанавливается")
        elif adjacent_xy(ch, tx, ty):                # act: вплотную — бьём
            try:
                log = c.fight(reason=reason)["result"]["combat_log"]
                print(f"  бой с {log['enemy']}: {log['outcome']} за {len(log['rounds'])} р. | xp+{log['xp_gained']} | лут {log['loot']}")
            except GameError as e:
                print("  бой не вышел:", e.code)      # напр. no_enemy_here — враг умер/ушёл
        else:                                         # act: шаг к цели
            c.move(*step_toward(ch, tx, ty), reason=reason)
        c.wait_cooldown()                             # wait

if WORLD_UP:
    run()

## 5. Проверка

Главный критерий M2 — **житель выжил** (hp > 0 и не в восстановлении). Базовый агент это проходит. Когда доведёшь `decide()`, цель та же, но уже на длинном прогоне: подними `rounds`, и агент должен **остаться живым** — отдыхать при низком hp и обходить волка.

In [ ]:
if WORLD_UP:
    ch = c.get_character()
    print("hp:", f'{ch["hp"]}/{ch["max_hp"]}', "| recovering:", ch["recovering"], "| xp:", ch["xp"], "| рюкзак:", ch["inventory"])
    assert ch["hp"] > 0 and not ch["recovering"], \
        "Житель погиб или восстанавливается. В этом и урок M2: береги hp и не лезь на сильных (см. Задачи)."
    if ch["xp"] > 0:
        print("✅ выжил и уже фармит бои — доведи decide() по Задачам 1–2 и подними rounds")
    else:
        print("✅ выжил. Боёв пока не засчитано (гоблин мог быть на респауне) — перезапусти ячейку run()")
else:
    print("Мир недоступен — проверка пропущена.")

## Наблюдаемость — смотри за умом своего бойца

Открой в браузере `BASE_URL/?token=<TOKEN>` (ссылка напечатана в разогреве): в одной вкладке крутится агент, в другой видно его шаги, **мысль-пузырь** (`reason`, который ты передаёшь в действия) и Хронику боёв. Это и есть «1 житель = 1 агент»: ты пишешь правила безопасности, а наблюдаешь живого бойца.

Ручной тык по API — `BASE_URL/docs` (кнопка **Authorize**, токен один раз без префикса `Bearer`).